## Scaled dot-product self-attention

The [simple attention mechanism](/aiml-common/lectures/nlp/transformers/transformers-intro/index) compares input vectors directly. Its compatibility function has no trainable parameters, although its input embeddings may already have been learned.

Consider "I love bears", "She bears the pain" and "Bears won the game". The word plays different grammatical and semantic roles.

How can the model learn which information to exchange between tokens? We separate the problem into deciding which tokens to read from and deciding what information to retrieve from them.

### Linear transformation of the input embeddings

Instead of comparing the input vectors directly, we learn three feature maps. These are not necessarily invertible changes of coordinates; they can reduce dimension and discard information. The purpose of these maps is to learn features that make contextually relevant tokens compatible, even when their original embeddings are dissimilar.

We define three trainable matrices $W^q, W^k, W^v$ and project each token, or equivalently the matrix $X$, into three different spaces:

$$
\begin{aligned}
Q &= XW^q \\
K &= XW^k \\
V &= XW^v
\end{aligned}
$$

Here $X$ is the $T \times d$ input embedding matrix, where $T$ is the input sequence length and $d$ is the embedding dimension. $Q$, $K$ and $V$ hold the queries, keys and values. The matrices $W^q, W^k, W^v$ have dimensions $d \times d_q$, $d \times d_k$ and $d \times d_v$. Queries and keys live in a space of the same dimension, so $d_q = d_k$.

For token $i$, the query $q_i$ is used to compare against keys $k_j$ from all allowed positions. The value $v_j$ is the information contributed by position $j$. A large query-key dot product gives a larger relative weight, all else equal. It is not necessarily a small Euclidean distance.

Consider "I love bears". The words `love` and `bears` are not semantically similar, but they participate in a verb-object relationship. Imagine a head whose query features express a request for a governing verb and whose key features indicate the ability to supply that information. The query of `bears` and the key of `love` could then align in the query-key space, even if their original embeddings were nearly orthogonal. In "She bears the pain", a different pattern could be useful: the verb `bears` can receive information from its subject `She` or, in bidirectional attention, its object `pain`.

These are linguistic analogies for learned features, not a claim that individual coordinates are labeled with grammatical roles. A head may learn such relationships through training, without explicit grammatical supervision. The value projection determines which features of the matched token are actually passed on.

The following toy example demonstrates the geometry with hand-designed feature maps.

### A geometric example: a noun looking for a verb

Use three orthogonal input vectors for `I`, `love` and `bears`. In this deliberately simplified example, the original embedding similarity of `bears` with `love` is zero. Choose two linear maps so that the query of `bears` and the key of `love` both point along the first axis of a two-dimensional feature space. Interpret that axis as a hypothetical verb-object compatibility feature.

The matrices below are constructed for illustration, not learned from text. In a real Transformer the same matrices are shared across positions and sentences and optimized through the training objective. A feature may be distributed across many coordinates.

In [ ]:
import torch

words = ["I", "love", "bears"]
X_demo = torch.eye(3)
Wq_demo = torch.tensor([[0., 1.], [0., 1.], [3., 0.]])
Wk_demo = torch.tensor([[0., 1.], [3., 0.], [0., 1.]])
q_demo = (X_demo @ Wq_demo)[2]
K_demo = X_demo @ Wk_demo
raw = X_demo[2] @ X_demo.T
projected = q_demo @ K_demo.T
print("Original scores:", raw)
print("Projected scores:", projected)
assert raw[1] == 0 and projected.argmax().item() == 1

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.quiver(0, 0, *q_demo.tolist(), angles="xy", scale_units="xy", scale=1, color="tab:blue", width=0.012, label="query: bears")
ax.quiver(0, 0, *K_demo[1].tolist(), angles="xy", scale_units="xy", scale=1, color="tab:orange", width=0.005, label="key: love (overlaps query)")
ax.quiver(0, 0, *K_demo[0].tolist(), angles="xy", scale_units="xy", scale=1, color="tab:green", label="keys: I, bears")
ax.set(xlim=(-0.3, 3.5), ylim=(-0.3, 3.5), xlabel="feature 1", ylabel="feature 2", aspect="equal")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
plt.show()

The original similarity favors the token itself. The transformed scores favor `love`. This demonstrates how separate query and key feature maps can change which relationships are emphasized. Alignment is sufficient here because the relevant vectors have equal norms; in general the dot product depends on both angle and magnitude.

### A controlled lookup example

Suppose a query for `country` is $(1,0)$, and the keys for `bear`, `through`, and `country` are $(3,0)$, $(0,1)$ and $(1,0)$. These are invented vectors, not measurements from a trained model. The query matches `bear` most strongly. If we change its value while keeping its key fixed, the weights remain unchanged but the retrieved information changes.

In [ ]:
import torch
import math
q = torch.tensor([1., 0.])
keys = torch.tensor([[3., 0.], [0., 1.], [1., 0.]])
values = torch.tensor([[2., 0.], [0., 1.], [1., 2.]])
a = torch.softmax(keys @ q / math.sqrt(2), dim=0)
print("weights (bear, through, country):", a)
print("output:", a @ values)
changed = values.clone(); changed[0] = torch.tensor([0., 4.])
print("output with same weights, changed value:", a @ changed)

In [ ]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False)

T, d, d_k = 8, 16, 4
X = torch.randn(T, d)  # input embeddings, one token per row

W_q = torch.randn(d, d_k) / math.sqrt(d)
W_k = torch.randn(d, d_k) / math.sqrt(d)
W_v = torch.randn(d, d_k) / math.sqrt(d)

Q, K, V = X @ W_q, X @ W_k, X @ W_v
print("Q:", tuple(Q.shape), " K:", tuple(K.shape), " V:", tuple(V.shape))

### Computation of the attention scores

Write the input matrix $X$ and the query weight matrix $W^q$ element by element (the key weight matrix is analogous):

$$
X = \begin{bmatrix}
x_{1,1} & x_{1,2} & \cdots & x_{1,d} \\
x_{2,1} & x_{2,2} & \cdots & x_{2,d} \\
\vdots & \vdots & \ddots & \vdots \\
x_{T,1} & x_{T,2} & \cdots & x_{T,d}
\end{bmatrix},
\qquad
W^q = \begin{bmatrix}
w_{1,1} & \cdots & w_{1,d_k} \\
\vdots & \ddots & \vdots \\
w_{d,1} & \cdots & w_{d,d_k}
\end{bmatrix}
$$

The product $Q = XW^q$ has dimensions $T \times d_k$. Each element $q_{i,j}$ of $Q$ is

$$
q_{i,j} = \sum_{l=1}^{d} x_{i,l} \, w_{l,j}.
$$

Each column $j$ of $W^q$ is a set of weights over the $d$ features of the input embedding. So the projection can weigh some features of a token more than others when it computes the token's query, and the same holds for the key.

The dot product now happens in the query-key space of dimension $d_k$. Treat $x_i$ as a row of $X$, a $1 \times d$ row vector. The query of token $i$ is $q_i = x_i W^q$ and the key of token $j$ is $k_j = x_j W^k$, so

$$
\begin{aligned}
q_i k_j^T &= x_i W^q (W^k)^T x_j^T = x_i W x_j^T, \\
W &= W^q (W^k)^T,
\end{aligned}
$$

where $W$ is a $d \times d$ matrix of rank at most $d_k$. Compared to the simple mechanism, the plain dot product $x_i x_j^T$ becomes a learned bilinear form. The next cell checks both statements numerically.

In [ ]:
W = W_q @ W_k.T  # d x d matrix of the bilinear form
print("Q K^T equals X W X^T:", torch.allclose(Q @ K.T, X @ W @ X.T, atol=1e-5))
print("W is", tuple(W.shape), "with rank", torch.linalg.matrix_rank(W).item())

Geometrically you can picture it as shown below.

![Queries and keys in the query-key space](images/self-attention-vectors.png)

*The diagram illustrates a possible alignment. The dot product depends on both vector norms and the angle between them. The query and keys only decide the attention weights. What is added to the token's output comes from the value vectors of the tokens it attends to.*

In matrix form, the scores are

$$
S = QK^T.
$$

### Scaling

The scores are divided by the square root of the key dimension, $\sqrt{d_k}$. Without this division, large score differences can cause the softmax to saturate: the largest score takes almost all of the weight and the weights of the other tokens shrink toward zero.

The experiment below shows this behavior. It applies the softmax to a vector of 8 scores, and then to the same vector multiplied by 100.

In [ ]:
def softmax(x):
    e_x = torch.exp(x - x.max())  # subtracting the max avoids overflow
    return e_x / e_x.sum()


scores = torch.tensor([0.171, 0.041, 0.079, 0.062, 0.192, 0.222, 0.115, 0.117])
print("softmax(scores):      ", softmax(scores))
print("softmax(100 * scores):", softmax(100 * scores))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, scale in zip(axes, (1, 100)):
    ax.bar(range(8), softmax(scale * scores), color="tab:green")
    ax.set_title(f"softmax({scale} x scores)" if scale != 1 else "softmax(scores)")
    ax.set_xlabel("token j")
axes[0].set_ylabel("attention weight")
plt.tight_layout()
plt.show()

With the original scores the weights are spread over all eight tokens. With the scores multiplied by 100, the token with the largest score takes most of the weight and the rest are close to zero. Under independent zero-mean unit-variance components, the score standard deviation grows as $\sqrt{d_k}$. Scaling controls this growth; it does not guarantee that softmax never saturates.

The [attention scaling](/aiml-common/lectures/nlp/transformers/attention-scaling-explained/attention_scaling_explained) section derives why $\sqrt{d_k}$ is the right factor, with a variance analysis and experiments.

The code for scaled dot-product attention is short. It works on batched queries and keys of shape $B \times T \times d_k$ and values of shape $B \times T \times d_v$. The optional mask is covered next.

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(dim_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value), weights


out, weights = scaled_dot_product_attention(Q[None], K[None], V[None])
print("output:", tuple(out.shape), " weights:", tuple(weights.shape))

### Masking and causal attention

A decoder generates one token at a time, so at inference a token never sees the tokens after it. During next-token training, the output at position $i$ predicts token $i+1$, and may see the input token at position $i$. Training has to respect this constraint. Otherwise the model would learn to use future tokens, which are ground truth that is not available during inference.

To prevent this, we _mask_ the scores of the future tokens by setting them to $-\infty$ before the softmax. The softmax output is then exactly 0 for those tokens. For query $q_i$, only keys $k_1$ through $k_i$ are accessible, and every key $k_j$ with $j > i$ is masked. For four tokens the following matrix shows the masking pattern (the factor $1/\sqrt{d_k}$ on finite scores is omitted for readability)

$$
\begin{array}{c|cccc}
 & k_1 & k_2 & k_3 & k_4 \\
\hline
q_1 & q_1 k_1^T & -\infty & -\infty & -\infty \\
q_2 & q_2 k_1^T & q_2 k_2^T & -\infty & -\infty \\
q_3 & q_3 k_1^T & q_3 k_2^T & q_3 k_3^T & -\infty \\
q_4 & q_4 k_1^T & q_4 k_2^T & q_4 k_3^T & q_4 k_4^T
\end{array}
$$

The mask is a lower-triangular matrix of ones. The next cell applies it to the 8-token context and compares the result with PyTorch's built-in causal attention.

In [ ]:
mask = torch.tril(torch.ones(T, T))
out, weights = scaled_dot_product_attention(Q[None], K[None], V[None], mask=mask)
print("causal attention weights:\n", weights[0])

reference = F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)
print("matches F.scaled_dot_product_attention:", torch.allclose(out, reference, atol=1e-6))

The first token can only attend to itself, so its row is a single 1. Every row still sums to 1, and every entry above the diagonal is 0.

### Softmax

The scores can be positive or negative. As in the simple mechanism, a softmax applied to each row, over the keys $j$ of each query $i$, turns them into the attention weights $\alpha_{ij}$. This is the `dim=-1` in the code above.

## Weighting the values

The attention weights form a weighted sum of the values, which is the output embedding:

$$
\hat v_i = \sum_{j=1}^T \alpha_{ij} v_j,
$$

where $\alpha_{ij}$ is the weight that query $i$ puts on token $j$, for an input sequence of length $T$.

Why have values at all, and why the extra matrix $W^v$? The values carry the information that a token uses to update its embedding. $W^v$ projects the input tokens to values, points in a $d_v$-dimensional space. Nothing requires $d_v = d_k$, but the two are usually equal. The value projection separates two jobs: the queries and keys decide the attention weights, and the values decide what each attended token contributes.

![Three grizzly bears on a riverbank in bear country](images/bear-country.jpg)

As an example, take the context "The hiking trail led us through _bear_ country". Suppose the query of `country` assigns a large weight to the key of `bear`. The value of `bear` then contributes to the output of `country`. That contribution is a learned vector, not necessarily an explicit representation of an animal or a noun. The residual connection and subsequent layers further transform the representation.

![Single attention head](images/scaled-dot-product-self-attention.png)

*A single attention head. Masking is not shown, and all vector spaces keep the same dimension.*

The full equation of scaled dot-product self-attention is

$$
\hat X = \mathtt{Attention}(Q, K, V) = \mathtt{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V,
$$

and the output has dimensions $T \times d_v$, where $T$ is the number of tokens in the input sequence and $d_v$ is the dimension of the values. With batching, every tensor gets a leading batch dimension.

<Warning>
Self-attention differs from a dense layer in where its weights come from. If a dense layer learns a very small weight for an input, it applies that weight to every example fed into the layer. In self-attention, a very small attention weight comes from the specific tokens present in that input, and a different input gives different weights.
</Warning>

## Example: contextual representations in BERT

We now return to the motivating example using a pretrained bidirectional encoder. Compare "I love bears", "She bears the pain" and "Bears won the game". The uncased BERT tokenizer maps `bears` to the same ID in all three sentences, and the code checks this. The token sits at a different position in each sentence, so its input embeddings already differ through the position embedding. We compare its hidden states across layers; these include attention, residual connections, normalization and feed-forward transformations.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from transformers import AutoModel, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()  # hide the weight-loading report
logging.disable_progress_bar()

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(
    checkpoint, output_attentions=True, output_hidden_states=True, attn_implementation="eager"
).eval()

sentences = ["I love bears", "She bears the pain", "Bears won the game"]
runs = []
for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])
    i = tokens.index("bears")
    runs.append({
        "tokens": tokens,
        "position": i,
        "token_id": inputs.input_ids[0, i].item(),
        # embedding of "bears": input embedding, then after each layer
        "embedding": torch.stack([h[0, i] for h in output.hidden_states]),
        # attention weights of "bears", averaged over the 12 heads of each layer
        "attention": torch.stack([a[0, :, i].mean(dim=0) for a in output.attentions]),
    })
    print(tokens, "-> 'bears' at position", i)

assert len({r["token_id"] for r in runs}) == 1
pairs = [(0, 1), (0, 2), (1, 2)]
similarities = [F.cosine_similarity(runs[a]["embedding"], runs[b]["embedding"], dim=-1) for a,b in pairs]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
labels = ["animal", "verb", "team"]
for (a,b), sim in zip(pairs, similarities):
    ax.plot(range(13), sim, marker="o", label=f"{labels[a]} / {labels[b]}")
ax.set(xlabel="layer (0 = input)", ylabel="cosine similarity", xticks=range(13))
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

At the input the three representations differ only through the position embedding. Differences in later layers demonstrate contextualization, not proof that a specific semantic sense has been isolated. Similarity need not decrease monotonically. The next figure shows attention weights averaged across heads; these are not a measure of causal importance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, run in zip(axes, runs):
    im = ax.imshow(run["attention"], cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(run["tokens"])), run["tokens"], rotation=45)
    ax.set_yticks(range(12), [str(l) for l in range(1, 13)])
    ax.set_title(" ".join(run["tokens"][1:-1]))
axes[0].set_ylabel("layer")
fig.colorbar(im, ax=axes, shrink=0.8, label="attention weight of 'bears'")
plt.show()

Inspect the maps for differences across contexts and layers. Averaging heads can hide distinct patterns; a large weight on a special token does not by itself establish its importance.

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Affine map of the input: $y = xA^T + b$. |
| [`nn.Softmax`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html) | Softmax over one dimension of a tensor. |
| [`nn.MultiheadAttention`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html) | Attention with several heads in parallel. |

Key references: [@Vaswani2017-vd; @Luong2015-dd]